In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("flats_moscow.csv", index_col=0)

df.head()

In [ ]:
df.info()
df.describe(include="all")

In [ ]:
df.isna().sum()
df.duplicated().sum()

In [ ]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

plt.figure(figsize=(14, 8))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, (len(numeric_cols) + 1) // 2, i)
    plt.hist(df[col], bins=30, edgecolor="black")
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="totsp", y="price")
plt.title("Зависимость цены от общей площади")
plt.show()

plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x="dist", y="price")
plt.title("Зависимость цены от расстояния до метро")
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="floor", y="price")
plt.title("Цена по этажам")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include=["int64", "float64"]).corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Корреляционная матрица числовых признаков")
plt.show()

In [ ]:
df_filtered = df.copy()

# пример: убрать слишком большие цены и площади
df_filtered = df_filtered[df_filtered["price"] < df_filtered["price"].quantile(0.99)]
df_filtered = df_filtered[df_filtered["totsp"] < df_filtered["totsp"].quantile(0.99)]

df_filtered.describe()

In [ ]:
cat_cols = df_filtered.select_dtypes(include=["object"]).columns
cat_cols

In [ ]:
df_encoded = pd.get_dummies(df_filtered, columns=cat_cols, drop_first=True)

df_encoded.head()

In [ ]:
df_encoded["price_per_m2"] = df_encoded["price"] / df_encoded["totsp"]
df_encoded["is_near_metro"] = (df_encoded["dist"] < df_encoded["dist"].median()).astype(int)

df_encoded[["price", "totsp", "price_per_m2", "dist", "is_near_metro"]].head()

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df_encoded["price_per_m2"], bins=30, kde=True)
plt.title("Распределение цены за м²")
plt.show()

In [ ]:
df_encoded.to_csv("flats_moscow_processed.csv")